In [4]:
import pandas as pd
import numpy as np
import os
import cv2
from ultralytics import YOLO
import shutil

# Загрузка CSV-файла
csv_path = "peper/homography_data.csv"
df = pd.read_csv(csv_path)

# Папка для сохранения аннотаций
output_annotations_dir = "peper/yolo_annotations"
os.makedirs(output_annotations_dir, exist_ok=True)

# Класс для листа бумаги (например, 0)
class_id = 0

# Функция для преобразования матрицы гомографии в координаты углов
def homography_to_corners(H, width, height):
    """
    Преобразует матрицу гомографии в координаты углов листа бумаги.

    Параметры:
        H (np.array): Матрица гомографии 3x3.
        width (int): Ширина изображения.
        height (int): Высота изображения.

    Возвращает:
        np.array: Координаты углов листа бумаги.
    """
    # Исходные углы (в однородных координатах)
    src_corners = np.array([
        [0, 0],  # Верхний левый
        [width, 0],  # Верхний правый
        [width, height],  # Нижний правый
        [0, height]  # Нижний левый
    ], dtype="float32")

    # Преобразуем углы с помощью матрицы гомографии
    dst_corners = cv2.perspectiveTransform(src_corners.reshape(-1, 1, 2), H)

    # Возвращаем координаты углов
    return dst_corners.reshape(4, 2)

# Обработка каждой строки в CSV
for index, row in df.iterrows():
    image_path = row['image_path']
    H = np.array([
        [row['h11'], row['h12'], row['h13']],
        [row['h21'], row['h22'], row['h23']],
        [row['h31'], row['h32'], 1.0]
    ], dtype="float32")

    # Загрузка изображения для получения размеров
    image = cv2.imread(image_path)
    if image is None:
        print(f"Ошибка: Не удалось загрузить изображение {image_path}.")
        continue
    height, width, _ = image.shape

    # Вычисляем обратную матрицу гомографии
    H_inv = np.linalg.inv(H)

    # Преобразуем матрицу гомографии в координаты углов
    corners = homography_to_corners(H_inv, width, height)
    print(f"Координаты углов для {image_path}:")
    print(corners)

    # Ограничиваем координаты углов рамками изображения
    corners[:, 0] = np.clip(corners[:, 0], 0, width)  # Ограничиваем x
    corners[:, 1] = np.clip(corners[:, 1], 0, height)  # Ограничиваем y

    # Вычисляем bounding box на основе ограниченных координат углов
    x_coords = [corner[0] for corner in corners]
    y_coords = [corner[1] for corner in corners]
    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)
    print(f"Bounding box: x_min={x_min}, x_max={x_max}, y_min={y_min}, y_max={y_max}")

    # Нормализация координат bounding box
    x_center = (x_min + x_max) / 2 / width
    y_center = (y_min + y_max) / 2 / height
    box_width = (x_max - x_min) / width
    box_height = (y_max - y_min) / height
    print(f"Normalized: x_center={x_center}, y_center={y_center}, width={box_width}, height={box_height}")

    # Сохраняем аннотацию в YOLO-формате
    annotation = f"{class_id} {x_center} {y_center} {box_width} {box_height}"
    annotation_file = os.path.join(output_annotations_dir, os.path.basename(image_path).replace(".png", ".txt").replace(".jpg", ".txt"))
    with open(annotation_file, "w") as f:
        f.write(annotation)

    print(f"Аннотация для {image_path} сохранена в {annotation_file}.")

# Создаем структуру данных для YOLOv8
dataset_dir = "datasets/peper"
os.makedirs(dataset_dir, exist_ok=True)

# Папки для изображений и аннотаций
images_train_dir = os.path.join(dataset_dir, "images", "train")
images_val_dir = os.path.join(dataset_dir, "images", "val")
labels_train_dir = os.path.join(dataset_dir, "labels", "train")
labels_val_dir = os.path.join(dataset_dir, "labels", "val")

# Создаем папки
os.makedirs(images_train_dir, exist_ok=True)
os.makedirs(images_val_dir, exist_ok=True)
os.makedirs(labels_train_dir, exist_ok=True)
os.makedirs(labels_val_dir, exist_ok=True)

# Разделяем данные на обучающую и валидационную выборки
image_files = [f for f in os.listdir(output_annotations_dir) if f.endswith(".txt")]
train_files = image_files[:int(0.8 * len(image_files))]  # 80% для обучения
val_files = image_files[int(0.8 * len(image_files)):]    # 20% для валидации

# Копируем изображения и аннотации в соответствующие папки
for file in train_files:
    image_name = file.replace(".txt", ".jpg")
    shutil.copy(os.path.join("peper/input_images", image_name), os.path.join(images_train_dir, image_name))
    shutil.copy(os.path.join(output_annotations_dir, file), os.path.join(labels_train_dir, file))

for file in val_files:
    image_name = file.replace(".txt", ".jpg")
    shutil.copy(os.path.join("peper/input_images", image_name), os.path.join(images_val_dir, image_name))
    shutil.copy(os.path.join(output_annotations_dir, file), os.path.join(labels_val_dir, file))

# Создаем конфигурационный файл для YOLOv8
yolo_config = f"""
path: {dataset_dir}  # Путь к датасету
train: images/train  # Папка с обучающими изображениями
val: images/val      # Папка с валидационными изображениями
nc: 1                # Количество классов (1 для листа бумаги)
names: ['paper']     # Имена классов
"""

# Сохраняем конфигурационный файл
with open("yolov8_config.yaml", "w") as f:
    f.write(yolo_config)

# Загрузка предобученной модели YOLOv8
model = YOLO("yolov8n.pt")

# Обучение модели
results = model.train(data="yolov8_config.yaml", epochs=50, imgsz=640)

# Использование обученной модели для детекции и выравнивания изображений
def align_image(image, bbox):
    """
    Выравнивает изображение на основе bounding box.

    Параметры:
        image (np.array): Исходное изображение.
        bbox (list): Координаты bounding box [x_min, y_min, x_max, y_max].

    Возвращает:
        np.array: Выровненное изображение.
    """
    x_min, y_min, x_max, y_max = bbox
    corners = np.array([
        [x_min, y_min],
        [x_max, y_min],
        [x_max, y_max],
        [x_min, y_max]
    ], dtype="float32")

    height, width = image.shape[:2]
    dst = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(corners, dst)
    aligned_image = cv2.warpPerspective(image, M, (width, height))
    return aligned_image

# Пример использования обученной модели
input_folder = "peper/extracted_frames"  # Папка с исходными изображениями
output_folder = "peper/aligned_images"   # Папка для сохранения выровненных изображений
os.makedirs(output_folder, exist_ok=True)

# Получаем список всех изображений в папке
image_files = [f for f in os.listdir(input_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]

# Обрабатываем каждое изображение
for image_file in image_files:
    input_path = os.path.join(input_folder, image_file)
    output_path = os.path.join(output_folder, f"aligned_{image_file}")

    # Загружаем и обрабатываем изображение
    try:
        # Загрузка изображения
        image = cv2.imread(input_path)
        if image is None:
            print(f"Ошибка: Не удалось загрузить изображение {input_path}.")
            continue

        # Детекция листа бумаги
        results = model(image)
        bbox = results[0].boxes.xyxy[0].cpu().numpy()  # Получаем bounding box

        # Выравнивание изображения
        aligned_image = align_image(image, bbox)

        # Сохранение результата
        cv2.imwrite(output_path, aligned_image)
        print(f"Выровненное изображение сохранено как {output_path}.")

        # Показываем результат
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax[0].set_title("Исходное изображение")
        ax[0].axis('off')

        ax[1].imshow(cv2.cvtColor(aligned_image, cv2.COLOR_BGR2RGB))
        ax[1].set_title("Выровненное изображение")
        ax[1].axis('off')

        plt.show()

    except Exception as e:
        print(f"Ошибка при обработке изображения {input_path}: {e}")

Координаты углов для peper/input_images\1.png:
[[         46          58]
 [        200          72]
 [        230         268]
 [         34         276]]
Bounding box: x_min=34.0, x_max=230.0, y_min=58.0, y_max=276.0
Normalized: x_center=0.5217391304347826, y_center=0.4704225352112676, width=0.7747035573122529, height=0.6140845070422535
Аннотация для peper/input_images\1.png сохранена в peper/yolo_annotations\1.txt.
Координаты углов для peper/input_images\1frame_0000.png:
[[        148         260]
 [        612         246]
 [        678         920]
 [        106         922]]
Bounding box: x_min=106.00001525878906, x_max=678.0, y_min=246.00001525878906, y_max=922.0
Normalized: x_center=0.5444444444444444, y_center=0.45625, width=0.7944444444444444, height=0.528125
Аннотация для peper/input_images\1frame_0000.png сохранена в peper/yolo_annotations\1frame_0000.txt.
Координаты углов для peper/input_images\1frame_0001.jpg:
[[        148         278]
 [        606         270]
 [      

FileNotFoundError: [Errno 2] No such file or directory: 'peper/input_images\\1.jpg'